# Белок + лиганд: MM/PBSA с OpenMM и APBS

Загрузите PDB белка и SDF/MOL2 лиганда с готовой позой в кармане. Блокнот подготовит параметры AmberTools, добавит воду и ионы, проведёт **5 пс MD**, сохранит **10 кадров** и рассчитает MM/PBSA.

Это учебная оценка без энтропийного вклада: 5 пс недостаточно для надёжной оценки свободной энергии связывания. Выполняйте ячейки сверху вниз в Google Colab.

**Убедитесь, что текущий Runtime использует GPU**


## 1. Входные файлы

- **Белок — PDB:** одна модель, стандартные аминокислоты, без лиганда и кофакторов. Пропущенные внутренние участки цепей нужно восстановить заранее; воду блокнот удалит.
- **Лиганд — SDF или MOL2:** одна органическая молекула, правильные порядки связей, протонирование и пространственные координаты в кармане. Ковалентные лиганды и металлокомплексы здесь не рассматриваются.

Исходную структуру комплекса можно взять из PDB или результата докинга. Белок и лиганд экспортируйте отдельно, **сохраняя общую систему координат**; при экспорте лиганда в формате SDF или MOL2 убедитесь, что порядок связей задан корректно. Водороды добавляются автоматически.


## 2. Установка

Запустите в новой сессии Colab. Устанавливаются OpenMM, AmberTools, APBS и FreeSASA.


In [ ]:
# Run in a fresh Google Colab session, before importing scientific packages.
import sys,subprocess,urllib.request,tarfile,io,os
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q',
                'openmm==8.3.1','parmed==4.3.1','pdbfixer==1.12.0',
                'rdkit','numpy','scipy','pandas','matplotlib','py3Dmol'],check=True)

# AmberTools CLI has a separate environment and does not replace Colab's Python.
AMBER_PREFIX=Path('/content/ambertools_env')
MAMBA_ROOT=Path('/content/mmpbsa_mamba')
MAMBA_EXE=Path('/content/mmpbsa_tools/micromamba')
MAMBA_EXE.parent.mkdir(exist_ok=True)
if not MAMBA_EXE.exists():
    url='https://micro.mamba.pm/api/micromamba/linux-64/latest'
    with urllib.request.urlopen(url,timeout=120) as response:
        payload=response.read()
    with tarfile.open(fileobj=io.BytesIO(payload),mode='r:bz2') as archive:
        member=archive.extractfile('bin/micromamba')
        if member is None:raise RuntimeError('В архиве нет micromamba')
        MAMBA_EXE.write_bytes(member.read())
    MAMBA_EXE.chmod(0o755)
if not all((AMBER_PREFIX/'bin'/name).is_file() for name in ['antechamber','parmchk2','tleap']):
    log=Path('/content/ambertools_install.log')
    with log.open('w') as f:
        result=subprocess.run([str(MAMBA_EXE),'create','-y','-r',str(MAMBA_ROOT),
                               '-p',str(AMBER_PREFIX),'-c','conda-forge',
                               'ambertools=24','python=3.11'],stdout=f,stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError('Не удалось установить AmberTools.\n'+log.read_text(errors='replace')[-6000:])
print('OpenMM и AmberTools готовы. Перезапуск ядра не требуется.')

# APBS has its own environment, independent of AmberTools and the Colab kernel.
APBS_PREFIX=Path('/content/apbs_env')
apbs_ready=(APBS_PREFIX/'bin/apbs').is_file() and (APBS_PREFIX/'bin/python').is_file()
if apbs_ready:
    apbs_ready=subprocess.run([str(APBS_PREFIX/'bin/python'),'-c','import freesasa'],
        stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode==0
if not apbs_ready:
    apbs_install_log=Path('/content/apbs_install.log')
    with apbs_install_log.open('w') as log:
        operation='install' if (APBS_PREFIX/'conda-meta').is_dir() else 'create'
        completed=subprocess.run([str(MAMBA_EXE),operation,'-y','-r',str(MAMBA_ROOT),
            '-p',str(APBS_PREFIX),'-c','conda-forge','apbs=3.4.1','freesasa-python=2.2.1','python=3.11'],
            stdout=log,stderr=subprocess.STDOUT)
    if completed.returncode:
        raise RuntimeError('Не удалось установить APBS.\n'+apbs_install_log.read_text(errors='replace')[-6000:])
print('APBS и FreeSASA готовы.')


## 3. Настройки

Используются ff14SB для белка, GAFF2/AM1-BCC для лиганда и TIP3P для воды. Нейтрализующие ионы добавляются автоматически; `EXTRA_SALT_PAIRS` задаёт дополнительные пары NaCl.

Динамика: 300 K, шаг 2 фс, 5 пс, 10 кадров. Для PB: диэлектрические проницаемости 1 внутри молекулы и 78.5 в растворителе, радиусы mbondi2. `PB_SALT_M` — концентрация соли в континуальной модели APBS, отдельно от ионов MD.


In [ ]:
import os,sys,json,io,subprocess,shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openmm as mm
from openmm import app,unit
import parmed as pmd
from pdbfixer import PDBFixer
from rdkit import Chem
from IPython.display import display
import py3Dmol
WORK=Path('mmpbsa_job').resolve(); WORK.mkdir(exist_ok=True)
PH=7.0
LIGAND_CHARGE=None  # None: formal charge from SDF/MOL2; or an integer override
PADDING_A=10.0
EXTRA_SALT_PAIRS=0  # neutralizing Na+/Cl- are always added when needed
TEMPERATURE_K=300.0
DT_PS=0.002
PRODUCTION_PS=5.0
N_FRAMES=10
SASA_SLICES=100
SASA_PROBE_NM=0.14
GAMMA_KCAL_A2=0.0054
BETA_KCAL=0.0
SEED=20261022
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
PB_SPACING_A=0.75
PB_PADDING_A=8.0
PB_EPS_OUT=78.5
PB_SALT_M=0.0
assert PB_SPACING_A>0 and PB_PADDING_A>=2 and PB_EPS_OUT>1 and PB_SALT_M>=0


## 4. Загрузка файлов

Выберите одновременно PDB белка и SDF/MOL2 лиганда.


In [ ]:
# Select exactly one protein PDB and one ligand SDF or MOL2 in the upload dialog.
from google.colab import files
uploaded=files.upload()
protein_names=[n for n in uploaded if n.lower().endswith('.pdb')]
ligand_names=[n for n in uploaded if n.lower().endswith(('.sdf','.mol2'))]
if len(protein_names)!=1 or len(ligand_names)!=1:
    raise ValueError('Загрузите один PDB белка и один SDF/MOL2 лиганда.')
protein_input=WORK/'protein_input.pdb'
ligand_input=WORK/('ligand_input'+Path(ligand_names[0]).suffix.lower())
protein_input.write_bytes(uploaded[protein_names[0]])
ligand_input.write_bytes(uploaded[ligand_names[0]])
print('Белок:',protein_names[0],'; лиганд:',ligand_names[0])

### Подготовка лиганда

Добавляем водороды, сохраняя координаты тяжёлых атомов.


In [ ]:
if ligand_input.suffix=='.sdf':
    supplier=Chem.SDMolSupplier(str(ligand_input),removeHs=False,sanitize=True)
    molecules=list(supplier)
    if len(molecules)!=1 or molecules[0] is None:
        raise ValueError('SDF должен содержать одну корректную молекулу.')
    ligand=molecules[0]
else:
    ligand=Chem.MolFromMol2File(str(ligand_input),removeHs=False,sanitize=True)
    if ligand is None:raise ValueError('RDKit не прочитал MOL2. Экспортируйте лиганд в SDF с порядками связей.')
if len(Chem.GetMolFrags(ligand))!=1 or ligand.GetNumConformers()!=1:
    raise ValueError('Нужна одна связная молекула с одной пространственной конформацией.')
if any(a.GetNumRadicalElectrons() for a in ligand.GetAtoms()):
    raise ValueError('Пример рассчитан на замкнутую электронную оболочку, без радикалов.')
if any(a.GetSymbol() not in {'H','C','N','O','F','P','S','Cl','Br','I'} for a in ligand.GetAtoms()):
    raise ValueError('Состав лиганда выходит за поддерживаемый здесь стандартный органический GAFF2-пример.')
ligand=Chem.AddHs(ligand,addCoords=True)
formal_charge=Chem.GetFormalCharge(ligand)
if LIGAND_CHARGE is not None and int(LIGAND_CHARGE)!=LIGAND_CHARGE:
    raise ValueError('LIGAND_CHARGE должен быть целым числом или None.')
net_charge=int(formal_charge if LIGAND_CHARGE is None else LIGAND_CHARGE)
if LIGAND_CHARGE is not None and net_charge!=formal_charge:
    raise ValueError('Сначала исправьте протонирование и формальные заряды в SDF/MOL2: указанный заряд не совпадает со структурой.')
with Chem.SDWriter(str(WORK/'ligand_with_H.sdf')) as writer:writer.write(ligand)
# Write only for visualization, never infer chemistry from this PDB.
for j,a in enumerate(ligand.GetAtoms()):
    info=Chem.AtomPDBResidueInfo()
    info.SetName((a.GetSymbol()+str(j+1)).rjust(4));info.SetResidueName('LIG')
    info.SetResidueNumber(1);info.SetChainId('L');info.SetIsHeteroAtom(True)
    a.SetMonomerInfo(info)
Chem.MolToPDBFile(ligand,str(WORK/'ligand_pose.pdb'))
print('Формальный заряд:',net_charge,'; атомов с H:',ligand.GetNumAtoms())


### Исходный комплекс

Белок показан голубой лентой, лиганд — палочками. Убедитесь, что лиганд находится в нужном кармане.


In [ ]:
initial_view=py3Dmol.view(width=900,height=450)
initial_view.addModel(protein_input.read_text(),'pdb')
initial_view.setStyle({'model':0},{'cartoon':{'color':'lightblue'}})
initial_view.addModel((WORK/'ligand_pose.pdb').read_text(),'pdb')
initial_view.setStyle({'model':1},{'stick':{'colorscheme':'orangeCarbon'},'sphere':{'scale':0.25}})
initial_view.zoomTo({'model':1});initial_view.zoom(0.6);initial_view.show()

## 5. Подготовка белка

PDBFixer добавляет недостающие атомы и водороды при заданном pH. Имена гистидинов, цистеинов в дисульфидных связях и концевых водородов приводятся к формату AMBER.


In [ ]:
if app.PDBFile(str(protein_input)).getNumFrames()!=1:
    raise ValueError('Выберите одну модель белка из многомодельного PDB перед загрузкой.')
fixer=PDBFixer(filename=str(protein_input))
removed=[(r.name,r.chain.id,r.id) for r in fixer.topology.residues()
         if r.name not in {'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
                           'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'}]
# Standard water is discarded; other species are reported rather than silently removed.
unsupported=[x for x in removed if x[0] not in {'HOH','WAT'}]
if unsupported:
    raise ValueError(f'В PDB есть нестандартные остатки/лиганд/ионы: {unsupported[:20]}. '
                     'Подайте PDB только стандартного белка; важные кофакторы требуют отдельной подготовки.')
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues()
chains=list(fixer.topology.chains())
internal={key:value for key,value in fixer.missingResidues.items()
          if key[1] not in (0,len(list(chains[key[0]].residues())))}
if internal:raise ValueError(f'Нужна предварительная модель внутренних пропусков белка: {internal}')
fixer.missingResidues={}  # Do not invent unresolved terminal extensions.
fixer.findNonstandardResidues()
if fixer.nonstandardResidues:raise ValueError('Нестандартные остатки нужно подготовить отдельно.')
fixer.findMissingAtoms();fixer.addMissingAtoms(seed=SEED)
# Without SEQRES, a gap may not be reported by findMissingResidues.
for chain in fixer.topology.chains():
    chain_res=list(chain.residues())
    for left,right in zip(chain_res[:-1],chain_res[1:]):
        la={a.name:a for a in left.atoms()};ra={a.name:a for a in right.atoms()}
        if 'C' in la and 'N' in ra:
            d=fixer.positions[la['C'].index]-fixer.positions[ra['N'].index]
            if np.linalg.norm(d.value_in_unit(unit.nanometer))>0.2:
                raise ValueError(f'Разрыв цепи между {left.id} и {right.id}: восстановите петлю '
                                 'или явно оформите независимые цепи через TER.')
fixer.topology.createDisulfideBonds(fixer.positions)
fixer.addMissingHydrogens(pH=PH)
residues=list(fixer.topology.residues());res_index={r:i+1 for i,r in enumerate(residues)}
disulfides=[]
for a,b in fixer.topology.bonds():
    if a.name==b.name=='SG' and a.residue!=b.residue:
        disulfides.append((res_index[a.residue],res_index[b.residue]))
        a.residue.name=b.residue.name='CYX'
for r in residues:
    if r.name=='HIS':
        names={a.name for a in r.atoms()}
        r.name='HIP' if {'HD1','HE2'}<=names else ('HID' if 'HD1' in names else 'HIE')
mod=app.Modeller(fixer.topology,fixer.positions)
mod.delete([a for a in mod.topology.atoms() if a.residue.name=='CYX' and a.name=='HG'])
# OpenMM uses H/H2/H3 at a protonated N terminus; LEaP expects H1/H2/H3.
for chain in mod.topology.chains():
    first=next(chain.residues(),None)
    if first is not None:
        named={a.name:a for a in first.atoms()}
        if {'H','H2','H3'}<=named.keys() and 'H1' not in named:
            named['H'].name='H1'
with open(WORK/'protein_prepared.pdb','w') as f:app.PDBFile.writeFile(mod.topology,mod.positions,f)
print('Остатков белка:',len(residues),'; дисульфидные связи (индексы LEaP):',disulfides)


## 6. Параметризация лиганда

`antechamber` назначает типы атомов GAFF2 и заряды AM1-BCC, записывая `ligand_gaff2.mol2`. `parmchk2` создаёт `ligand.frcmod` с дополнительными параметрами. Эти файлы затем читает LEaP.


In [ ]:
AMBER_PREFIX=Path(AMBER_PREFIX).resolve()
amber_env=os.environ.copy();amber_env['AMBERHOME']=str(AMBER_PREFIX)
amber_env['PATH']=str(AMBER_PREFIX/'bin')+os.pathsep+amber_env.get('PATH','')
amber_env['LD_LIBRARY_PATH']=str(AMBER_PREFIX/'lib')+os.pathsep+amber_env.get('LD_LIBRARY_PATH','')
def run_amber(tool,args,log_name):
    command=[str(AMBER_PREFIX/'bin'/tool)]+list(args)
    with open(WORK/log_name,'w') as f:
        result=subprocess.run(command,cwd=WORK,env=amber_env,stdout=f,stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f'{tool}: ошибка. См. {log_name}\n'+(WORK/log_name).read_text(errors='replace')[-5000:])
run_amber('antechamber',['-i','ligand_with_H.sdf','-fi','mdl','-o','ligand_gaff2.mol2',
                         '-fo','mol2','-at','gaff2','-c','bcc','-nc',str(net_charge),
                         '-m','1','-rn','LIG','-s','2','-pf','y'],'antechamber.log')
run_amber('parmchk2',['-i','ligand_gaff2.mol2','-f','mol2','-o','ligand.frcmod','-s','gaff2'],
          'parmchk2.log')
if 'ATTN' in (WORK/'ligand.frcmod').read_text().upper():
    raise RuntimeError('Есть параметры ATTN/NEEDS REVISION в ligand.frcmod. Нужна ручная проверка.')
print('Параметры GAFF2 и заряды AM1-BCC готовы.')


## 7. Комплекс, вода и ионы

LEaP объединяет белок и лиганд, добавляет слой воды 10 Å и ионы. Файлы `solvated.prmtop` и `solvated.inpcrd` содержат параметры и координаты системы для OpenMM. `dry.prmtop` и отдельные топологии белка и лиганда нужны для MM/PBSA без явной воды.


In [ ]:
if EXTRA_SALT_PAIRS<0 or int(EXTRA_SALT_PAIRS)!=EXTRA_SALT_PAIRS:
    raise ValueError('EXTRA_SALT_PAIRS должен быть неотрицательным целым.')
leap_lines=['source leaprc.protein.ff14SB','source leaprc.gaff2','source leaprc.water.tip3p',
            'set default PBRadii mbondi2','loadAmberParams ligand.frcmod',
            'lig = loadMol2 ligand_gaff2.mol2','rec = loadPdb protein_prepared.pdb']
leap_lines += [f'bond rec.{a}.SG rec.{b}.SG' for a,b in disulfides]
leap_lines += ['complex = combine { rec lig }','check complex',
              'saveAmberParm complex dry.prmtop dry.inpcrd','savePdb complex dry.pdb',
              f'solvateBox complex TIP3PBOX {PADDING_A}',
              'addIonsRand complex Na+ 0','addIonsRand complex Cl- 0']
if EXTRA_SALT_PAIRS:
    leap_lines += [f'addIonsRand complex Na+ {int(EXTRA_SALT_PAIRS)}',
                   f'addIonsRand complex Cl- {int(EXTRA_SALT_PAIRS)}']
leap_lines += ['check complex','saveAmberParm complex solvated.prmtop solvated.inpcrd',
              'savePdb complex solvated.pdb','quit']
(WORK/'build.leap').write_text('\n'.join(leap_lines)+'\n')
for name in ['dry.prmtop','dry.inpcrd','dry.pdb','solvated.prmtop','solvated.inpcrd','solvated.pdb']:
    (WORK/name).unlink(missing_ok=True)  # A failed rerun must not reuse old output.
run_amber('tleap',['-f','build.leap'],'tleap.log')
for name in ['dry.prmtop','dry.inpcrd','solvated.prmtop','solvated.inpcrd']:
    if not (WORK/name).is_file() or (WORK/name).stat().st_size==0:
        raise RuntimeError('LEaP не создал '+name+'; проверьте tleap.log')
dry=pmd.load_file(str(WORK/'dry.prmtop'),xyz=str(WORK/'dry.inpcrd'))
solvated=pmd.load_file(str(WORK/'solvated.prmtop'),xyz=str(WORK/'solvated.inpcrd'))
lig_ids=np.array([a.idx for a in dry.atoms if a.residue.name=='LIG'])
prot_ids=np.array([a.idx for a in dry.atoms if a.residue.name!='LIG'])
if len(lig_ids)!=ligand.GetNumAtoms():raise RuntimeError('Число атомов лиганда изменилось в LEaP.')
print('Атомов комплекса:',len(dry.atoms),'; с водой и ионами:',len(solvated.atoms))
# Derive all three dry topologies from a single parent. No independent minimization.
for key,mask in [('protein','!:LIG'),('ligand',':LIG')]:
    # Rebuild type tables to remove unused parent LJ types after stripping.
    subset=dry[mask].copy(pmd.Structure)
    compact=pmd.amber.AmberParm.from_structure(subset)
    compact.save(str(WORK/(key+'.prmtop')),overwrite=True)


## 8. Минимизация и динамика OpenMM

После минимизации запускается NVT с термостатом Ланжевена. Минимизация не входит в 5 пс. Длительность и число кадров одинаковы на GPU и CPU.


In [ ]:
solv_prm=app.AmberPrmtopFile(str(WORK/'solvated.prmtop'))
solv_crd=app.AmberInpcrdFile(str(WORK/'solvated.inpcrd'))
md_system=solv_prm.createSystem(nonbondedMethod=app.PME,nonbondedCutoff=1.0*unit.nanometer,
                              constraints=app.HBonds,rigidWater=True,ewaldErrorTolerance=1e-4)
platform=None;properties={}
for name,props in [('CUDA',{'Precision':'mixed'}),('OpenCL',{'Precision':'mixed'}),('CPU',{'Threads':'2'})]:
    try:
        candidate=mm.Platform.getPlatformByName(name)
        integrator=mm.LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin,1/unit.picosecond,DT_PS*unit.picosecond)
        integrator.setRandomNumberSeed(SEED)
        md=mm.Context(md_system,integrator,candidate,props)
        platform,properties=candidate,props
        break
    except Exception:
        continue
if platform is None:raise RuntimeError('Не удалось создать контекст OpenMM на GPU или CPU.')
md.setPositions(solv_crd.positions)
md.setPeriodicBoxVectors(*solv_crd.boxVectors)
mm.LocalEnergyMinimizer.minimize(md,10,500)
md.setVelocitiesToTemperature(TEMPERATURE_K*unit.kelvin,SEED)
md.applyVelocityConstraints(1e-6)
print('Платформа:',platform.getName(),'; модель: ff14SB / GAFF2 / TIP3P / PME')


### 10 кадров за 5 пс

Кадры сохраняются каждые 0.5 пс. Для анализа оставляем белок и лиганд; координаты не сворачиваем по отдельным атомам через границы ячейки.


In [ ]:
n_steps=round(PRODUCTION_PS/DT_PS)
stride=n_steps//N_FRAMES
frames=[];md_energies=[]
for frame_no in range(1,N_FRAMES+1):
    integrator.step(stride)
    state=md.getState(getPositions=True,getEnergy=True,enforcePeriodicBox=False)
    xyz=state.getPositions(asNumpy=True).value_in_unit(unit.nanometer)[:len(dry.atoms)]
    if not np.isfinite(xyz).all():raise RuntimeError('Нестабильная траектория')
    frames.append(np.array(xyz))
    ep=state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    ek=state.getKineticEnergy().value_in_unit(unit.kilojoule_per_mole)
    md_energies.append({'frame':frame_no,'time_ps':frame_no*stride*DT_PS,
                        'potential_kJ_mol':ep,'kinetic_kJ_mol':ek,'total_kJ_mol':ep+ek})
    if frame_no%10==0:print(f'MD: сохранён кадр {frame_no}/{N_FRAMES}')
frames=np.asarray(frames);md_energies=pd.DataFrame(md_energies)
np.savez_compressed(WORK/'solute_trajectory.npz',positions_nm=frames,
                    time_ps=md_energies.time_ps.to_numpy())
md_energies.to_csv(WORK/'md_energies.csv',index=False)
print('Сохранены 10 кадров за 5 пс; вода и ионы исключены из массива анализа.')
del md,integrator


## 9. MM/PBSA: формулы

Для каждого кадра используем координаты комплекса $C$, белка $P$ и лиганда $L$ из одной траектории:

$$
\Delta G_{\mathrm{MM/PBSA}}\approx\Delta E_{\mathrm{MM}}+\Delta G_{\mathrm{PB}}+\Delta G_{\mathrm{np}},
\qquad \Delta X=X_C-X_P-X_L.
$$

$E_{\mathrm{MM}}$ — вакуумная энергия OpenMM; при таком выборе координат внутренние связанные члены сокращаются. На графиках отдельно показаны взаимодействия ван-дер-Ваальса и кулоновский вклад.

APBS решает линейное уравнение Пуассона–Больцмана. Полярная энергия сольватации — разность электростатических энергий в растворителе и однородном референсе с $\varepsilon=1$:

$$
G_{\mathrm{PB}}=E_{\mathrm{APBS}}^{\mathrm{solvent}}-E_{\mathrm{APBS}}^{\mathrm{reference}}.
$$

Обе задачи, а также $C$, $P$ и $L$, используют общую сетку. Заряды и радиусы берутся из AMBER-топологии.

Неполярный вклад рассчитывается по площади поверхности, доступной растворителю (SASA), с помощью FreeSASA:

$$
G_{\mathrm{np}}=\gamma\,\mathrm{SASA}+\beta,\qquad
\gamma=0.0054\ \mathrm{kcal\,mol^{-1}\,\mathring{A}^{-2}},\quad\beta=0.
$$

Энтропийная поправка $-T\Delta S$ не включена. Результат — приближённая энергетическая оценка, а не абсолютная свободная энергия связывания.


In [ ]:
import re
import numpy as np
from pathlib import Path
K_E=138.93545764438198  # kJ mol^-1 nm e^-2

def make_pb_grid(trajectory_nm,radii_A,spacing_A=0.75,padding_A=8.0):
    """One fixed numerical domain for all frames and all three components."""
    xyz=np.asarray(trajectory_nm)*10.0
    lower=np.min(xyz-np.asarray(radii_A)[None,:,None],axis=(0,1))-padding_A
    upper=np.max(xyz+np.asarray(radii_A)[None,:,None],axis=(0,1))+padding_A
    center=(lower+upper)/2
    dime=(32*np.maximum(2,np.ceil((upper-lower)/spacing_A/32).astype(int))+1)
    fine=(dime-1)*spacing_A
    return {'dime':dime.tolist(),'fglen':fine.tolist(),
            'cglen':(1.7*fine).tolist(),'center':center.tolist()}

def write_pqr(path,xyz_nm,charges,radii_A,atoms=None):
    xyz=np.asarray(xyz_nm)*10.0
    if not (len(xyz)==len(charges)==len(radii_A)):raise ValueError('Несовпадение размеров PQR')
    if not np.isfinite(xyz).all() or not np.isfinite(charges).all() or np.any(np.asarray(radii_A)<=0):
        raise ValueError('Некорректные координаты/заряды/радиусы PQR')
    lines=[]
    for i,((x,y,z),q,radius) in enumerate(zip(xyz,charges,radii_A)):
        name=atoms[i].name if atoms is not None else 'I'
        res=atoms[i].residue.name if atoms is not None else 'ION'
        resid=atoms[i].residue.idx+1 if atoms is not None else 1
        lines.append(f'ATOM {i+1:6d} {name:4s} {res:3s} {resid:5d} '
                     f'{x:.6f} {y:.6f} {z:.6f} {q:.10f} {radius:.6f}')
    Path(path).write_text('\n'.join(lines)+'\n')

def run_apbs_polar(xyz_nm,charges,radii_A,grid,folder,atoms=None,
                   eps_out=78.5,salt_M=0.0,temperature_K=300.0):
    """Return solvent-minus-reference electrostatic charging energy in kJ/mol."""
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    write_pqr(folder/'solute.pqr',xyz_nm,charges,radii_A,atoms)
    fmt=lambda a:' '.join(f'{float(v):.7f}' for v in a)
    lines=['read',' mol pqr solute.pqr','end']
    for name,sdie,ionic_strength in [('solvent',eps_out,salt_M),('reference',1.0,0.0)]:
        lines += [f'elec name {name}',' mg-auto',
                  ' dime '+' '.join(str(int(n)) for n in grid['dime']),
                  ' cglen '+fmt(grid['cglen']),' fglen '+fmt(grid['fglen']),
                  ' cgcent '+fmt(grid['center']),' fgcent '+fmt(grid['center']),
                  ' mol 1',' lpbe',' bcfl mdh',' pdie 1.0',f' sdie {sdie:.8f}',
                  ' srfm smol',' chgm spl2',' srad 1.4',' swin 0.3',' sdens 10.0',
                  f' temp {temperature_K:.6f}']
        if ionic_strength:
            lines += [f' ion charge 1 conc {ionic_strength:.8f} radius 2.0',
                      f' ion charge -1 conc {ionic_strength:.8f} radius 2.0']
        lines += [' calcenergy total',' calcforce no','end']
    lines += ['print elecEnergy solvent end','print elecEnergy reference end',
              'print elecEnergy solvent - reference end','quit']
    (folder/'apbs.in').write_text('\n'.join(lines)+'\n')
    env=os.environ.copy();env['OMP_NUM_THREADS']='1'
    env['LD_LIBRARY_PATH']=str(Path(APBS_PREFIX)/'lib')+os.pathsep+env.get('LD_LIBRARY_PATH','')
    completed=subprocess.run([str(Path(APBS_PREFIX)/'bin/apbs'),'apbs.in'],
                             cwd=folder,env=env,stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT,text=True)
    output=completed.stdout
    # Write after APBS exits; keep stdout separate from any solver-owned log files.
    (folder/'apbs_stdout.log').write_text(output)
    if completed.returncode:
        raise RuntimeError(f'APBS завершился с ошибкой, см. {folder}/apbs_stdout.log\n'+output[-4000:])
    found=re.findall(r'Global net ELEC energy\s*=\s*([-+0-9.EeDd]+)\s+kJ/mol',output)
    if len(found)!=3:
        raise RuntimeError(f'Ожидались три PRINT-энергии APBS; получено {len(found)}. См. {folder}/apbs_stdout.log')
    solv,ref,polar=[float(s.replace('D','E').replace('d','e')) for s in found]
    if re.search(r'not converg|failed to converge|exceeded.*iterations',output,re.I):
        raise RuntimeError('APBS сообщает о несходимости решателя; проверьте журнал.')
    return {'PB_kJ_mol':polar,'solvent_charging_kJ_mol':solv,'reference_charging_kJ_mol':ref}


# Use the prebuilt FreeSASA Python binding in the APBS environment.
# No compilation or Python-kernel replacement is needed in Colab.
FREESASA_WORKER=r'''
import json,sys,freesasa
payload=json.load(sys.stdin)
parameters=freesasa.Parameters({'algorithm':freesasa.LeeRichards,
    'probe-radius':payload['probe_A'],'n-slices':payload['n_slices'],'n-threads':1})
result=freesasa.calcCoord(payload['coords_A'],payload['radii_A'],parameters)
json.dump([result.atomArea(i) for i in range(len(payload['radii_A']))],sys.stdout)
'''

def sasa_freesasa(xyz_nm,radii_nm,probe_nm=0.14,n_slices=100):
    """FreeSASA Lee-Richards through its Python API; per-atom SASA in nm^2."""
    payload={'coords_A':(np.asarray(xyz_nm,dtype=float)*10).ravel().tolist(),
             'radii_A':(np.asarray(radii_nm,dtype=float)*10).tolist(),
             'probe_A':probe_nm*10,'n_slices':int(n_slices)}
    env=os.environ.copy();env.pop('PYTHONPATH',None);env.pop('PYTHONHOME',None)
    env['LD_LIBRARY_PATH']=str(Path(APBS_PREFIX)/'lib')+os.pathsep+env.get('LD_LIBRARY_PATH','')
    completed=subprocess.run([str(Path(APBS_PREFIX)/'bin/python'),'-c',FREESASA_WORKER],
                             input=json.dumps(payload),text=True,capture_output=True,env=env)
    if completed.returncode:
        raise RuntimeError(f'Ошибка FreeSASA (код {completed.returncode}):\n'+completed.stderr[-4000:])
    areas=np.asarray(json.loads(completed.stdout))/100.0
    if len(areas)!=len(radii_nm) or not np.isfinite(areas).all():raise RuntimeError('Некорректный ответ FreeSASA')
    return areas


### Подготовка расчёта энергий

Создаём вакуумные системы OpenMM и общую сетку APBS для всех кадров.


In [ ]:
prms={key:app.AmberPrmtopFile(str(WORK/file)) for key,file in
      {'complex':'dry.prmtop','protein':'protein.prmtop','ligand':'ligand.prmtop'}.items()}
idx={'complex':np.arange(len(dry.atoms)),'protein':prot_ids,'ligand':lig_ids}
mm_eval={};pb_params={}
for key,prm in prms.items():
    vac=prm.createSystem(nonbondedMethod=app.NoCutoff,constraints=None,removeCMMotion=False)
    if any(isinstance(f,(mm.CustomNonbondedForce,mm.CustomBondForce)) for f in vac.getForces()):
        raise RuntimeError('Специальные nonbonded-члены не поддержаны в данном LJ-разложении.')
    for force in vac.getForces():force.setForceGroup(1 if isinstance(force,mm.NonbondedForce) else 0)
    ii=mm.VerletIntegrator(.001);cc=mm.Context(vac,ii,platform,properties)
    mm_eval[key]=(vac,ii,cc)
    parent_atoms=[dry.atoms[int(i)] for i in idx[key]]
    pb_params[key]={'charges':np.array([a.charge for a in parent_atoms]),
                    'radii_A':np.array(dry.parm_data['RADII'])[idx[key]],'atoms':parent_atoms}
    if np.any(pb_params[key]['radii_A']<=0):raise ValueError('Нужны положительные радиусы PQR')
parent_nb=next(f for f in mm_eval['complex'][0].getForces() if isinstance(f,mm.NonbondedForce))
pb_grid=make_pb_grid(frames,pb_params['complex']['radii_A'],PB_SPACING_A,PB_PADDING_A)
(WORK/'pb_grid.json').write_text(json.dumps(pb_grid,indent=2))
print('Общая PB-сетка:',pb_grid['dime'],'; шаг, Å:',
      np.array(pb_grid['fglen'])/(np.array(pb_grid['dime'])-1))
print('Параметры и 1–4-исключения согласованы; PQR использует исходные AMBER-заряды и радиусы.')


## 10. Анализ 10 кадров

Для каждого кадра рассчитываем энергии комплекса, белка и лиганда, затем их разности. Итоговые вклады приводятся в kcal/mol; SD показывает разброс кадров.


In [ ]:
q,sig,eps=[],[],[]
for j in range(parent_nb.getNumParticles()):
    a,b,c=parent_nb.getParticleParameters(j)
    q.append(a.value_in_unit(unit.elementary_charge));sig.append(b.value_in_unit(unit.nanometer))
    eps.append(c.value_in_unit(unit.kilojoule_per_mole))
q,sig,eps=map(np.asarray,(q,sig,eps))
rows=[];component_rows=[]
for frame_no,xyz in enumerate(frames,1):
    components={}
    for key in prms:
        sub=xyz[idx[key]];ctx=mm_eval[key][2];ctx.setPositions(sub)
        bonded=ctx.getState(getEnergy=True,groups={0}).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
        nb_energy=ctx.getState(getEnergy=True,groups={1}).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
        params=pb_params[key]
        pb_result=run_apbs_polar(sub,params['charges'],params['radii_A'],pb_grid,
            WORK/f'apbs_frames/frame_{frame_no:03d}/{key}',atoms=params['atoms'],
            eps_out=PB_EPS_OUT,salt_M=PB_SALT_M,temperature_K=TEMPERATURE_K)
        polar=pb_result['PB_kJ_mol']
        area=sasa_freesasa(sub,params['radii_A']/10.0,SASA_PROBE_NM,SASA_SLICES).sum()
        nonpolar=(GAMMA_KCAL_A2*area*100+BETA_KCAL)*4.184
        components[key]=np.array([bonded,nb_energy,polar,nonpolar])
        component_rows.append({'frame':frame_no,'component':key,'MM_kcal_mol':(bonded+nb_energy)/4.184,
                               'PB_kcal_mol':polar/4.184,
                               'SASA_A2':area*100,'nonpolar_kcal_mol':nonpolar/4.184})
    delta=(components['complex']-components['protein']-components['ligand'])/4.184
    r=np.linalg.norm(xyz[prot_ids,None]-xyz[None,lig_ids],axis=-1)
    sigma=(sig[prot_ids,None]+sig[None,lig_ids])/2
    epsilon=np.sqrt(eps[prot_ids,None]*eps[None,lig_ids]);sr6=(sigma/r)**6
    lj=np.sum(4*epsilon*(sr6**2-sr6))/4.184
    coul=K_E*np.sum(q[prot_ids,None]*q[None,lig_ids]/r)/4.184
    rows.append({'frame':frame_no,'time_ps':frame_no*stride*DT_PS,'vdW':lj,'Coulomb':coul,
                 'PB':delta[2],'nonpolar':delta[3],'MM':delta[0]+delta[1],
                 'MMPBSA':delta.sum()})
    if frame_no%5==0:print(f'MM/PBSA: обработан кадр {frame_no}/{N_FRAMES}')
results=pd.DataFrame(rows);component_results=pd.DataFrame(component_rows)
results.to_csv(WORK/'mmpbsa_10_frames.csv',index=False)
component_results.to_csv(WORK/'component_energies.csv',index=False)
cols=['vdW','Coulomb','PB','nonpolar','MMPBSA']
summary=pd.DataFrame({'mean_kcal_mol':results[cols].mean(),'SD_frames_kcal_mol':results[cols].std()})
summary.to_csv(WORK/'mmpbsa_summary.csv');display(summary.round(3))
print('SD — разброс коррелированных кадров, а не ошибка предсказания аффинности.')


## 11. Графики

Показаны вклады и сумма MM/PBSA, их средние значения и энергия всей системы MD. В NVT полная механическая энергия флуктуирует из-за обмена с термостатом.


In [ ]:
fig,ax=plt.subplots(2,2,figsize=(13,8))
for col in ['vdW','Coulomb','PB','nonpolar']:ax[0,0].plot(results.frame,results[col],label=col)
ax[0,0].set(title='Вклады в MM/PBSA',xlabel='Номер кадра',ylabel='kcal/mol');ax[0,0].legend()
ax[0,1].plot(results.frame,results.MMPBSA,'o-',ms=3,label='MM/PBSA')
ax[0,1].plot(results.frame,results.MMPBSA.expanding().mean(),lw=2,label='Накопительное среднее')
ax[0,1].set(title='Суммарный MM/PBSA score',xlabel='Номер кадра',ylabel='kcal/mol');ax[0,1].legend()
for col,label in [('potential_kJ_mol','Потенциальная'),('kinetic_kJ_mol','Кинетическая'),('total_kJ_mol','Полная')]:
    ax[1,0].plot(md_energies.frame,md_energies[col],label=label)
ax[1,0].set(title='Вся система: белок + лиганд + вода + ионы',xlabel='Номер кадра',ylabel='kJ/mol');ax[1,0].legend()
ax[1,1].bar(cols,summary.mean_kcal_mol,yerr=summary.SD_frames_kcal_mol,capsize=3)
ax[1,1].axhline(0,color='k',lw=.6);ax[1,1].set(title='Средние вклады ± SD кадров',ylabel='kcal/mol')
fig.tight_layout();fig.savefig(WORK/'mmpbsa_contributions.png',dpi=160);plt.show()

## 12. Анимация

Кадры совмещаются по Cα белка только для отображения. В расчёте энергий используются исходные координаты.


In [ ]:
ca=np.array([a.idx for a in dry.atoms if a.name=='CA' and a.residue.name!='LIG'])
if len(ca)<3:raise ValueError('Нужно хотя бы три Cα для совмещения анимации.')
ref=frames[0];aligned=[]
for xyz in frames:
    c=xyz[ca].mean(0);c0=ref[ca].mean(0)
    u,s,vt=np.linalg.svd((xyz[ca]-c).T@(ref[ca]-c0))
    rotation=u@np.diag([1,1,np.linalg.det(u@vt)])@vt
    aligned.append((xyz-c)@rotation+c0)
text=[]
for k,xyz in enumerate(aligned,1):
    buffer=io.StringIO();app.PDBFile.writeFile(prms['complex'].topology,xyz*unit.nanometer,buffer)
    body='\n'.join(line for line in buffer.getvalue().splitlines() if line.startswith(('ATOM','HETATM','TER')))
    text.append(f'MODEL     {k:4d}\n{body}\nENDMDL\n')
trajectory_pdb=''.join(text);(WORK/'solute_aligned_10_frames.pdb').write_text(trajectory_pdb)
view=py3Dmol.view(width=900,height=450);view.addModelsAsFrames(trajectory_pdb,'pdb')
view.setStyle({'cartoon':{'color':'lightblue'}})
view.setStyle({'resn':'LIG'},{'stick':{'colorscheme':'orangeCarbon'}})
view.zoomTo({'resn':'LIG'});view.zoom(0.65)
view.animate({'loop':'forward','interval':150});view.show()

## 13. Сохранение результатов

Архив содержит входные файлы, подготовленные топологии, 10 кадров, таблицы вкладов, график и файлы расчётов APBS. Для скачивания задайте `DOWNLOAD_RESULTS=True`.


In [ ]:
import importlib.metadata
run_info={'production_ps':PRODUCTION_PS,'n_frames':N_FRAMES,'dt_ps':DT_PS,
          'temperature_K':TEMPERATURE_K,'protein_pH':PH,'water_padding_A':PADDING_A,
          'random_seed':SEED,'platform':platform.getName(),'ligand_charge':net_charge,
          'force_fields':['ff14SB','GAFF2/AM1-BCC','TIP3P'],
          'polar_model':'APBS LPBE; solvent minus homogeneous reference',
          'pqr_radii':'AMBER mbondi2','solute_dielectric':1.0,'solvent_dielectric':PB_EPS_OUT,
          'PB_salt_M':PB_SALT_M,'PB_surface':'smol','PB_charge_mapping':'spl2',
          'PB_boundary':'mdh','PB_method':'mg-auto','PB_grid':pb_grid,
          'PB_probe_A':1.4,
          'nonpolar':'gamma*SASA + beta; FreeSASA Lee-Richards',
          'gamma_kcal_mol_A2':GAMMA_KCAL_A2,'beta_kcal_mol':BETA_KCAL,
          'SASA_slices':SASA_SLICES,'SASA_probe_nm':SASA_PROBE_NM,
          'entropy_correction':False,'extra_salt_pairs':EXTRA_SALT_PAIRS,
          'versions':{p:importlib.metadata.version(p) for p in ['openmm','parmed','rdkit','pdbfixer','numpy','scipy']}}
for package,prefix in [('ambertools',AMBER_PREFIX),('apbs',APBS_PREFIX),('freesasa-python',APBS_PREFIX)]:
    meta=list((Path(prefix)/'conda-meta').glob(package+'-*.json'))
    if meta:run_info['versions'][package]=json.loads(meta[0].read_text()).get('version')
(WORK/'run_info.json').write_text(json.dumps(run_info,indent=2))
archive=shutil.make_archive(str(WORK.parent/'mmpbsa_job_results'),'zip',WORK)
print('Архив результатов:',archive)
DOWNLOAD_RESULTS=False
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)
